# Preprocess the data & save

Input schema:  
`[symbol, date, open, high, low, close, volume_usd]`


In [220]:
import pandas as pd
import numpy as np

In [221]:
df = pd.read_csv('../data/coins.csv')
df.head()

,unix,date,symbol,open,high,low,close,volume_usd
0,1523937600,2018-04-17 04:00:00,ADA/USD,0.25551,0.28800,0.25551,0.26664,2165077
1,1523941200,2018-04-17 05:00:00,ADA/USD,0.26660,0.27798,0.26010,0.26200,2235633
2,1523944800,2018-04-17 06:00:00,ADA/USD,0.26221,0.26396,0.24800,0.25664,2153964
3,1523948400,2018-04-17 07:00:00,ADA/USD,0.25662,0.26300,0.25489,0.25698,1215621
4,1523952000,2018-04-17 08:00:00,ADA/USD,0.25636,0.25998,0.25229,0.25631,896096


In [222]:
df.shape

(844335, 8)

In [223]:
df['date'] = pd.to_datetime(df['date'])

In [224]:
print(f'Min: {df["date"].min()}\nMax: {df["date"].max()}')

Min: 2017-08-17 04:00:00
Max: 2023-10-19 23:00:00


In [225]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 844335 entries, 0 to 844334
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   unix        844335 non-null  int64         
 1   date        844335 non-null  datetime64[ns]
 2   symbol      844335 non-null  object        
 3   open        844335 non-null  float64       
 4   high        844335 non-null  float64       
 5   low         844335 non-null  float64       
 6   close       844335 non-null  float64       
 7   volume_usd  844335 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(2), object(1)
memory usage: 51.5+ MB


## token extraction (BTC, ETH)


In [226]:
# Feature extraction; we do not intend to trade these
# symbols because the market is too efficient.
btc = df[df['symbol'] == 'BTC/USD']
eth = df[df['symbol'] == 'ETH/USD']

# Keep only close price for calculations
to_drop = ['open', 'high', 'low', 'volume_usd', 'symbol']
btc = btc[['date', 'close']].rename(columns={'close': 'btc_close'})
eth = eth[['date', 'close']].rename(columns={'close': 'eth_close'})

btc.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54116 entries, 239332 to 293447
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       54116 non-null  datetime64[ns]
 1   btc_close  54116 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 1.2 MB


In [227]:
print(len(df['symbol'].unique()))
df = df[~df['symbol'].isin({'ETH/USD', 'BTC/USD'})]
print(len(df['symbol'].unique()))

21
19


## next_close


In [228]:
# Create the target feature
df['next_close'] = df.groupby('symbol')['close'].shift(-1)
df.head()

,unix,date,symbol,open,high,low,close,volume_usd,next_close
0,1523937600,2018-04-17 04:00:00,ADA/USD,0.25551,0.28800,0.25551,0.26664,2165077,0.26200
1,1523941200,2018-04-17 05:00:00,ADA/USD,0.26660,0.27798,0.26010,0.26200,2235633,0.25664
2,1523944800,2018-04-17 06:00:00,ADA/USD,0.26221,0.26396,0.24800,0.25664,2153964,0.25698
3,1523948400,2018-04-17 07:00:00,ADA/USD,0.25662,0.26300,0.25489,0.25698,1215621,0.25631
4,1523952000,2018-04-17 08:00:00,ADA/USD,0.25636,0.25998,0.25229,0.25631,896096,0.25536


## OHLCV_log (log returns)


In [229]:
log_returns = lambda s: np.log(s / s.shift(1))  # noqa: E731
df['open_log'] = df.groupby('symbol')['open'].transform(log_returns)
df['high_log'] = df.groupby('symbol')['high'].transform(log_returns)
df['low_log'] = df.groupby('symbol')['low'].transform(log_returns)
df['close_log'] = df.groupby('symbol')['close'].transform(log_returns)

# The previous next close is the current close price so we can reverse by:
# predicted_next_close = current_close * exp(predicted_next_close_log)
df['next_close_log'] = df.groupby('symbol')['next_close'].transform(log_returns)

# Volume log uses a different formula simply to shrink the range
df['volume_log'] = df.groupby('symbol')['volume_usd'].transform(lambda x: np.log(1 + x))

In [230]:
df.head()

,unix,date,symbol,open,high,low,close,volume_usd,next_close,open_log,high_log,low_log,close_log,next_close_log,volume_log
0,1523937600,2018-04-17 04:00:00,ADA/USD,0.25551,0.28800,0.25551,0.26664,2165077,0.26200,NaN,NaN,NaN,NaN,NaN,14.587967
1,1523941200,2018-04-17 05:00:00,ADA/USD,0.26660,0.27798,0.26010,0.26200,2235633,0.25664,0.042488,-0.035411,0.017805,-0.017555,-0.020670,14.620035
2,1523944800,2018-04-17 06:00:00,ADA/USD,0.26221,0.26396,0.24800,0.25664,2153964,0.25698,-0.016604,-0.051752,-0.047637,-0.020670,0.001324,14.582821
3,1523948400,2018-04-17 07:00:00,ADA/USD,0.25662,0.26300,0.25489,0.25698,1215621,0.25631,-0.021549,-0.003644,0.027403,0.001324,-0.002611,14.010766
4,1523952000,2018-04-17 08:00:00,ADA/USD,0.25636,0.25998,0.25229,0.25631,896096,0.25536,-0.001014,-0.011549,-0.010253,-0.002611,-0.003713,13.705804


## add year, eth, btc


In [231]:
# Normalize year to [0, 1] scale for every 10 years
# This allows the model to extrapolate to future years naturally
df['year'] = (df['date'].dt.year - 2010) / 10

In [232]:
df = df.merge(btc, on='date', how='inner')
df = df.merge(eth, on='date', how='inner')

In [233]:
df['eth_btc_ratio'] = df['eth_close'] / df['btc_close']

In [234]:
df['btc_close_log'] = df.groupby('symbol')['btc_close'].transform(log_returns)
df['eth_close_log'] = df.groupby('symbol')['eth_close'].transform(log_returns)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736103 entries, 0 to 736102
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   unix            736103 non-null  int64         
 1   date            736103 non-null  datetime64[ns]
 2   symbol          736103 non-null  object        
 3   open            736103 non-null  float64       
 4   high            736103 non-null  float64       
 5   low             736103 non-null  float64       
 6   close           736103 non-null  float64       
 7   volume_usd      736103 non-null  int64         
 8   next_close      736084 non-null  float64       
 9   open_log        736084 non-null  float64       
 10  high_log        736084 non-null  float64       
 11  low_log         736084 non-null  float64       
 12  close_log       736084 non-null  float64       
 13  next_close_log  736065 non-null  float64       
 14  volume_log      736103 non-null  flo

In [235]:
df.head()

,unix,date,symbol,open,high,low,close,volume_usd,next_close,open_log,...,low_log,close_log,next_close_log,volume_log,year,btc_close,eth_close,eth_btc_ratio,btc_close_log,eth_close_log
0,1523937600,2018-04-17 04:00:00,ADA/USD,0.25551,0.28800,0.25551,0.26664,2165077,0.26200,NaN,...,NaN,NaN,NaN,14.587967,0.8,8030.00,507.60,0.063213,NaN,NaN
1,1523941200,2018-04-17 05:00:00,ADA/USD,0.26660,0.27798,0.26010,0.26200,2235633,0.25664,0.042488,...,0.017805,-0.017555,-0.020670,14.620035,0.8,7983.00,505.56,0.063330,-0.005870,-0.004027
2,1523944800,2018-04-17 06:00:00,ADA/USD,0.26221,0.26396,0.24800,0.25664,2153964,0.25698,-0.016604,...,-0.047637,-0.020670,0.001324,14.582821,0.8,8025.01,507.77,0.063273,0.005249,0.004362
3,1523948400,2018-04-17 07:00:00,ADA/USD,0.25662,0.26300,0.25489,0.25698,1215621,0.25631,-0.021549,...,0.027403,0.001324,-0.002611,14.010766,0.8,8138.45,514.28,0.063191,0.014037,0.012739
4,1523952000,2018-04-17 08:00:00,ADA/USD,0.25636,0.25998,0.25229,0.25631,896096,0.25536,-0.001014,...,-0.010253,-0.002611,-0.003713,13.705804,0.8,8131.60,516.42,0.063508,-0.000842,0.004153


## drop null-containing rows


In [236]:
# Drop all null-containing rows
df.dropna(inplace=True, ignore_index=True)

## time_idx


In [237]:
# check to make sure everything is non-null
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736065 entries, 0 to 736064
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   unix            736065 non-null  int64         
 1   date            736065 non-null  datetime64[ns]
 2   symbol          736065 non-null  object        
 3   open            736065 non-null  float64       
 4   high            736065 non-null  float64       
 5   low             736065 non-null  float64       
 6   close           736065 non-null  float64       
 7   volume_usd      736065 non-null  int64         
 8   next_close      736065 non-null  float64       
 9   open_log        736065 non-null  float64       
 10  high_log        736065 non-null  float64       
 11  low_log         736065 non-null  float64       
 12  close_log       736065 non-null  float64       
 13  next_close_log  736065 non-null  float64       
 14  volume_log      736065 non-null  flo

In [238]:
df['time_idx'] = df.groupby('symbol').cumcount()

# Cyclical features


In [239]:
PI = np.pi
df.sample(10)

,unix,date,symbol,open,high,low,close,volume_usd,next_close,open_log,...,close_log,next_close_log,volume_log,year,btc_close,eth_close,eth_btc_ratio,btc_close_log,eth_close_log,time_idx
280750,1611500400,2021-01-24 15:00:00,DENT/USD,0.000293,0.000296,0.000292,0.000295,14314,0.000295,-0.001365,...,0.006129,0.000000,9.569063,1.1,31957.05,1320.50,0.041321,-0.003083,0.012145,12373
572021,1649937600,2022-04-14 12:00:00,MKR/USD,1875.000000,1888.000000,1873.000000,1886.000000,82471,1917.000000,-0.004258,...,0.005850,0.016303,11.320214,1.2,41149.98,3086.95,0.075017,0.005605,0.004601,15099
551803,1679320800,2023-03-20 14:00:00,MATIC/USD,1.140200,1.146000,1.131600,1.144500,6004221,1.131500,-0.011510,...,0.003852,-0.011424,15.607973,1.3,28055.89,1774.25,0.063240,0.008345,0.007916,34122
531154,1604919600,2020-11-09 11:00:00,MATIC/USD,0.016650,0.016660,0.016370,0.016490,157289,0.016070,0.016349,...,-0.008454,-0.025800,11.965847,1.0,15802.39,458.25,0.028999,0.021528,0.012937,13473
552902,1683280800,2023-05-05 10:00:00,MATIC/USD,0.988000,0.989500,0.986700,0.989300,932382,0.991700,0.000101,...,0.001315,0.002423,13.745499,1.3,29110.68,1904.51,0.065423,0.000333,0.002439,35221
290959,1648299600,2022-03-26 13:00:00,DENT/USD,0.002797,0.002808,0.002785,0.002798,216029,0.002790,-0.006770,...,-0.000715,-0.002863,12.283173,1.2,44251.67,3112.30,0.070332,-0.002220,-0.003800,22582
155028,1581501600,2020-02-12 10:00:00,BCH/USD,470.160000,474.860000,464.440000,473.930000,4593059,472.840000,-0.008619,...,0.008135,-0.002303,15.340057,1.0,10305.83,254.30,0.024675,0.002903,0.004809,1822
606005,1644069600,2022-02-05 14:00:00,RVN/USD,0.076410,0.076990,0.076050,0.076190,222338,0.076310,0.008147,...,-0.002883,0.001574,12.311959,1.2,41458.73,3021.49,0.072879,-0.001253,-0.000523,20703
625501,1613656800,2021-02-18 14:00:00,SOL/USD,8.429500,9.033300,8.421500,9.015100,3893537,9.241000,-0.005407,...,0.067021,0.024749,15.174829,1.1,52233.43,1924.02,0.036835,0.008405,0.005467,4585
6464,1547312400,2019-01-12 17:00:00,ADA/USD,0.042450,0.042730,0.042300,0.042610,198294,0.042550,-0.005872,...,0.003998,-0.001409,12.197511,0.9,3592.00,124.25,0.034591,0.000585,0.003225,6464


## hour


In [240]:
hour = lambda h: (2 * PI * h) / 24  # noqa: E731

df['hour_sin'] = np.sin(hour(df['date'].dt.hour))
df['hour_cos'] = np.cos(hour(df['date'].dt.hour))

## weekday


In [241]:
weekday = lambda w: (2 * PI * w) / 7  # noqa: E731

df['weekday_sin'] = np.sin(weekday(df['date'].dt.dayofweek))
df['weekday_cos'] = np.cos(weekday(df['date'].dt.dayofweek))

## month


In [242]:
month = lambda m: (2 * PI * m) / 12  # noqa: E731

df['month_sin'] = np.sin(month(df['date'].dt.month))
df['month_cos'] = np.cos(month(df['date'].dt.month))

## normalized day


In [243]:
norm_day = lambda nd: (2 * PI * nd)  # noqa: E731

df['norm_day_sin'] = np.sin(norm_day(df['date'].dt.day / df['date'].dt.daysinmonth))
df['norm_day_cos'] = np.cos(norm_day(df['date'].dt.day / df['date'].dt.daysinmonth))

## drop cols used for feature extraction


In [244]:
# drop columns used for feature extraction
to_drop = [
  'open',
  'high',
  'low',
  'close',
  'volume_usd',
  'next_close',
  'btc_close',
  'eth_close',
  'date',
  'unix',
]
df = df.drop(columns=to_drop)
df.columns

Index(['symbol', 'open_log', 'high_log', 'low_log', 'close_log',
       'next_close_log', 'volume_log', 'year', 'eth_btc_ratio',
       'btc_close_log', 'eth_close_log', 'time_idx', 'hour_sin', 'hour_cos',
       'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'norm_day_sin',
       'norm_day_cos'],
      dtype='object')

In [245]:
col_order = [
  'time_idx',
  'symbol',
  'open_log',
  'high_log',
  'low_log',
  'close_log',
  'volume_log',
  'btc_close_log',
  'eth_close_log',
  'eth_btc_ratio',
  'hour_sin',
  'hour_cos',
  'norm_day_sin',
  'norm_day_cos',
  'weekday_sin',
  'weekday_cos',
  'month_sin',
  'month_cos',
  'year',
  'next_close_log',
]
df = df.reindex(columns=col_order)

In [246]:
df.head()

,time_idx,symbol,open_log,high_log,low_log,close_log,volume_log,btc_close_log,eth_close_log,eth_btc_ratio,hour_sin,hour_cos,norm_day_sin,norm_day_cos,weekday_sin,weekday_cos,month_sin,month_cos,year,next_close_log
0,0,ADA/USD,0.042488,-0.035411,0.017805,-0.017555,14.620035,-0.005870,-0.004027,0.063330,0.965926,2.588190e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,-0.020670
1,1,ADA/USD,-0.016604,-0.051752,-0.047637,-0.020670,14.582821,0.005249,0.004362,0.063273,1.000000,6.123234e-17,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,0.001324
2,2,ADA/USD,-0.021549,-0.003644,0.027403,0.001324,14.010766,0.014037,0.012739,0.063191,0.965926,-2.588190e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,-0.002611
3,3,ADA/USD,-0.001014,-0.011549,-0.010253,-0.002611,13.705804,-0.000842,0.004153,0.063508,0.866025,-5.000000e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,-0.003713
4,4,ADA/USD,-0.000195,-0.000693,0.002810,-0.003713,13.441326,0.001574,0.003151,0.063608,0.707107,-7.071068e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,0.003986


In [247]:
df.shape

(736065, 20)

## Save after preprocessing done

(This takes awhile so it's commented out)


In [ ]:
# df.to_csv('../data/preproc_coins.csv', index= False)